!pip -q install groq streamlit

In [ ]:
!pip -q install -U groq streamlit pyngrok

#Groq API Key

In [ ]:
from getpass import getpass
import os
from groq import Groq

GROQ_API_KEY = getpass("Enter your Groq API Key: ").strip()
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
MODEL_ID = "openai/gpt-oss-20b"
client = Groq(api_key=GROQ_API_KEY)

print("✅ Groq API configured and client initialized")


## API test

The test uses the supported Groq model ID `openai/gpt-oss-20b`. Older Llama SpecDec model IDs have been retired. Run the API key setup cell first.

In [ ]:
# Recreate the client from the key cell if this cell is run independently.
import os
from groq import Groq

if not os.environ.get("GROQ_API_KEY"):
    raise RuntimeError("Run the Groq API key setup cell first.")
client = Groq(api_key=os.environ["GROQ_API_KEY"])

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": "Write a one-line introduction for Yashwanth K R, a second-year Information Science & Engineering student focused on Java and backend development."
        }
    ],
    temperature=0.4
)

print(response.choices[0].message.content)


#Create the student's data

In [ ]:
import json

student = {
    "name": "Yashwanth K R",
    "degree": "B.E. in Information Science & Engineering",
    "college": "Don Bosco Institute of Technology",
    "year": "2nd Year",
    "study_period": "2025–2029",
    "location": "Bengaluru, India",
    "career_goal": "Backend & Java Developer",
    "about": (
        "I have been curious about how the universe works since childhood, "
        "especially space science, cosmology, celestial mechanics, and astronomy. "
        "That interest shapes how I approach engineering: I like to understand "
        "why systems work, from JVM internals and memory models to backend APIs "
        "and distributed systems. I am focused on Java engineering, OOP, systems "
        "design, and AI engineering, and enjoy building clean, reliable software."
    ),
    "core_domains": [
        "Core & Advanced Java",
        "Backend & RESTful APIs",
        "OOP & System Architecture",
        "Hardware & Embedded Systems",
        "Space Tech & WebGL"
    ],
    "skill_groups": {
        "Backend & Architecture": [
            "Java (Core & Advanced)", "OOP & Design Patterns", "Node.js & Express",
            "REST API Engineering", "Razorpay Payment Systems", "Python & FastAPI",
            "JWT & Authentication", "Webhook Security & HMAC"
        ],
        "Data, Cloud & Infrastructure": [
            "SQL / Relational DBs", "MongoDB", "Linux & Unix Shell", "Docker & Minikube",
            "Git Version Control", "Networking & APIs", "Arduino & Embedded C++", "Bluetooth & IoT"
        ],
        "Frontend & Web Technologies": [
            "React.js", "Next.js", "TypeScript / ES6+", "Three.js / WebGL",
            "Tailwind CSS", "Computer Vision (YOLO)", "C# / .NET (Aspire)"
        ]
    },
    "projects": [
        {"title": "Spotify Clone", "url": "https://spotify-ggx2.onrender.com/"},
        {"title": "Cinemate", "url": "https://cinemate-wine.vercel.app/"}
    ],
    "links": {
        "Portfolio": "https://yash2006kr.vercel.app/#home",
        "GitHub": "https://github.com/yash2006kr",
        "LinkedIn": "https://www.linkedin.com/in/yash2006kr",
        "LeetCode": "https://leetcode.com/u/yash2006kr",
        "X / Twitter": "https://x.com/yash2006kr",
        "Resume": "https://yash2006kr.vercel.app/resume.pdf"
    },
    "email": "yashwanthkr@outlook.com"
}

print(json.dumps(student, indent=2, ensure_ascii=False))

#Create the AI content generator

In [ ]:
import json

def generate_portfolio_content(student):
    skill_summary = "\n".join(
        f"{group}: {', '.join(skills)}"
        for group, skills in student["skill_groups"].items()
    )
    project_summary = "\n".join(
        f"{project['title']} — {project['url']}"
        for project in student["projects"]
    )
    profile = f"""
Name: {student['name']}
Education: {student['degree']} at {student['college']}
Year: {student['year']} ({student['study_period']})
Location: {student['location']}
Career focus: {student['career_goal']}
About/background: {student['about']}
Core domains: {', '.join(student['core_domains'])}
Technical skills:
{skill_summary}
Portfolio projects (titles and links only; project details are not provided):
{project_summary}
"""
    prompt = f"""
You are an accurate portfolio copy editor. Use only the supplied profile.
Do not invent grades, internships, certifications, project features, metrics, or achievements.
The project entries only provide names and URLs, so do not infer how they work.
Return only valid JSON with this structure:
{{
  "tagline": "",
  "about": "",
  "career_goal": "",
  "project_descriptions": [
    {{"title": "", "description": "", "technologies": []}}
  ],
  "career_message": ""
}}
Write an 80–120 word first-person About section, a concise recruiter-friendly tagline,
and one generic sentence per project that invites readers to explore its linked build
without claiming features. Keep the career message factual and concise.

PROFILE:
{profile}
"""
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0.3
    )
    text = response.choices[0].message.content
    try:
        return json.loads(text)
    except (TypeError, json.JSONDecodeError) as exc:
        print("Could not parse the model response:", exc)
        print(text)
        return None

#Generate the content

In [ ]:
portfolio = generate_portfolio_content(student)

print(
    json.dumps(
        portfolio,
        indent=2
    )
)

#Create the actual website

In [ ]:
%%writefile app.py

import json
import os
import streamlit as st
from groq import Groq

st.set_page_config(page_title="Yashwanth K R | Portfolio", page_icon="🪐", layout="wide")

student = {
    "name": "Yashwanth K R",
    "degree": "B.E. in Information Science & Engineering",
    "college": "Don Bosco Institute of Technology",
    "year": "2nd Year",
    "study_period": "2025–2029",
    "location": "Bengaluru, India",
    "career_goal": "Backend & Java Developer",
    "about": "I have been curious about how the universe works since childhood, especially space science, cosmology, celestial mechanics, and astronomy. That interest shapes how I approach engineering: I like to understand why systems work, from JVM internals and memory models to backend APIs and distributed systems. I am focused on Java engineering, OOP, systems design, and AI engineering, and enjoy building clean, reliable software.",
    "core_domains": ["Core & Advanced Java", "Backend & RESTful APIs", "OOP & System Architecture", "Hardware & Embedded Systems", "Space Tech & WebGL"],
    "skill_groups": {
        "Backend & Architecture": ["Java (Core & Advanced)", "OOP & Design Patterns", "Node.js & Express", "REST API Engineering", "Razorpay Payment Systems", "Python & FastAPI", "JWT & Authentication", "Webhook Security & HMAC"],
        "Data, Cloud & Infrastructure": ["SQL / Relational DBs", "MongoDB", "Linux & Unix Shell", "Docker & Minikube", "Git Version Control", "Networking & APIs", "Arduino & Embedded C++", "Bluetooth & IoT"],
        "Frontend & Web Technologies": ["React.js", "Next.js", "TypeScript / ES6+", "Three.js / WebGL", "Tailwind CSS", "Computer Vision (YOLO)", "C# / .NET (Aspire)"]
    },
    "projects": [
        {"title": "Spotify Clone", "url": "https://spotify-ggx2.onrender.com/"},
        {"title": "Cinemate", "url": "https://cinemate-wine.vercel.app/"}
    ],
    "links": {
        "Portfolio": "https://yash2006kr.vercel.app/#home",
        "GitHub": "https://github.com/yash2006kr",
        "LinkedIn": "https://www.linkedin.com/in/yash2006kr",
        "LeetCode": "https://leetcode.com/u/yash2006kr",
        "X / Twitter": "https://x.com/yash2006kr",
        "Resume": "https://yash2006kr.vercel.app/resume.pdf"
    },
    "email": "yashwanthkr@outlook.com"
}

api_key = os.environ.get("GROQ_API_KEY")
if not api_key:
    st.error("GROQ_API_KEY is not configured. Run the key setup cell before launching the app.")
    st.stop()
MODEL_ID = "openai/gpt-oss-20b"
client = Groq(api_key=api_key)

DEFAULT_CONTENT = {
    "tagline": "Backend & Java Developer • Building real-world systems, exploring space tech, and growing as an engineer every day.",
    "about": student["about"],
    "career_goal": student["career_goal"],
    "career_message": "Focused on understanding systems deeply and building reliable software with curiosity and care."
}


def generate_content():
    skills = "\n".join(f"{name}: {', '.join(values)}" for name, values in student["skill_groups"].items())
    project_names = ", ".join(project["title"] for project in student["projects"])
    prompt = f"""
You are a careful portfolio copy editor. Write concise JSON fields tagline, about,
career_goal, career_message for this student. Only use facts below. Do not invent
CGPA, internship, certifications, project features, metrics, or achievements.
About must be first-person and 80-120 words. Project names are {project_names};
do not describe their functionality because no descriptions were supplied.
Name: {student['name']}
Education: {student['degree']} at {student['college']}, {student['year']} ({student['study_period']})
Location: {student['location']}
Career focus: {student['career_goal']}
Current About: {student['about']}
Core domains: {', '.join(student['core_domains'])}
Skills:\n{skills}
Return only a JSON object with string fields tagline, about, career_goal, career_message.
"""
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0.3
    )
    content = json.loads(response.choices[0].message.content)
    for field in DEFAULT_CONTENT:
        if not isinstance(content.get(field), str) or not content[field].strip():
            content[field] = DEFAULT_CONTENT[field]
    return content

if "portfolio_content" not in st.session_state:
    st.session_state.portfolio_content = DEFAULT_CONTENT.copy()

st.sidebar.title("Portfolio Studio")
st.sidebar.caption("Profile details are based on the supplied portfolio.")
if st.sidebar.button("✨ Refine my portfolio copy", type="primary"):
    with st.spinner("Polishing the profile with AI…"):
        try:
            st.session_state.portfolio_content = generate_content()
            st.sidebar.success("Portfolio copy updated.")
        except Exception as exc:
            st.sidebar.error(f"Could not generate copy: {exc}")

content = st.session_state.portfolio_content
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Space+Grotesk:wght@500;600;700&display=swap');
.stApp { background:#f5f6f8; color:#17202b; font-family:'DM Sans',sans-serif; }
.block-container { max-width:1100px; padding-top:2rem; }
.hero { background:linear-gradient(130deg,#111827,#24245b 65%,#3d286b); color:#fff; padding:42px; border-radius:24px; margin:10px 0 26px; }
.hero .eyebrow { color:#c4b5fd; text-transform:uppercase; letter-spacing:.16em; font-size:.75rem; font-weight:700; }
.hero h1 { font-family:'Space Grotesk',sans-serif; font-size:clamp(2.5rem,6vw,4rem); margin:12px 0 5px; }
.hero .sub { font-size:1.1rem; color:#e0e7ff; }
.card { background:#fff; border:1px solid #e4e7ec; border-radius:18px; padding:22px; margin:10px 0; }
.pill { display:inline-block; margin:4px 5px 4px 0; padding:7px 12px; background:#eef2ff; color:#3730a3; border-radius:20px; font-size:.88rem; }
a { color:#4338ca!important; }
</style>
""", unsafe_allow_html=True)

st.markdown(f"""
<div class="hero">
  <div class="eyebrow">Information Science · Systems · Curiosity</div>
  <h1>{student['name']}</h1>
  <div class="sub">{content['tagline']}</div>
  <p>{student['degree']} · {student['college']} · {student['location']}</p>
</div>
""", unsafe_allow_html=True)

st.header("About")
st.write(content["about"])
st.info(f"Career focus: {content['career_goal']}")

st.header("Core domains")
st.markdown(" ".join(f'<span class="pill">{domain}</span>' for domain in student["core_domains"]), unsafe_allow_html=True)

st.header("Technical skills & stack")
for group, skills in student["skill_groups"].items():
    with st.expander(group, expanded=True):
        st.markdown(" ".join(f'<span class="pill">{skill}</span>' for skill in skills), unsafe_allow_html=True)

st.header("Creations")
columns = st.columns(len(student["projects"]))
for column, project in zip(columns, student["projects"]):
    with column:
        st.markdown(f"<div class='card'><h3>{project['title']}</h3><p>Explore the live project from my portfolio.</p><a href='{project['url']}' target='_blank'>Open project ↗</a></div>", unsafe_allow_html=True)

st.header("Academic orbit")
st.markdown(f"**{student['college']}**  \n{student['degree']}  \n{student['study_period']} · {student['location']}  \nSpecializing in Data Structures, Operating Systems, Database Engineering, and Networks.")
st.write(content["career_message"])

st.header("Connect")
st.markdown(f"Email: [{student['email']}](mailto:{student['email']})")
link_columns = st.columns(3)
for i, (label, url) in enumerate(student["links"].items()):
    with link_columns[i % 3]:
        st.markdown(f"[{label} ↗]({url})")

st.divider()
st.caption(f"© 2026 {student['name']} · Java · Node.js · React · Three.js")


#Start the website

In [ ]:
import os
import subprocess
import sys
import time
import urllib.error
import urllib.request

# Stop a server started by an earlier run of this cell, if there is one.
previous_process = globals().get("portfolio_streamlit_process")
if previous_process is not None and previous_process.poll() is None:
    previous_process.terminate()
    try:
        previous_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        previous_process.kill()

log_handle = open("/content/portfolio.log", "w")
portfolio_streamlit_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--browser.gatherUsageStats", "false",
    ],
    cwd="/content",
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

# Wait for Streamlit to answer before proceeding to the Colab proxy link.
for attempt in range(45):
    if portfolio_streamlit_process.poll() is not None:
        log_handle.flush()
        log_text = open("/content/portfolio.log", encoding="utf-8", errors="replace").read()
        raise RuntimeError("Streamlit exited during startup. Log:\n" + log_text[-6000:])
    try:
        with urllib.request.urlopen("http://127.0.0.1:8501", timeout=2) as response:
            if response.status == 200:
                print("✅ Portfolio app is responding on port 8501")
                break
    except (urllib.error.URLError, TimeoutError):
        time.sleep(1)
else:
    log_handle.flush()
    log_text = open("/content/portfolio.log", encoding="utf-8", errors="replace").read()
    raise TimeoutError("Portfolio did not start. Log:\n" + log_text[-6000:])

log_handle.flush()


In [ ]:
import urllib.error
import urllib.request

try:
    with urllib.request.urlopen("http://127.0.0.1:8501", timeout=5) as response:
        print("Local app status:", response.status)
        print("Portfolio server is ready for the ngrok tunnel.")
except urllib.error.URLError as exc:
    print("Portfolio server is not responding:", exc)
    try:
        print(open("/content/portfolio.log", encoding="utf-8", errors="replace").read()[-6000:])
    except OSError:
        print("No Streamlit log was created.")


In [ ]:
# Expose the Streamlit portfolio using an ngrok public URL.
# Get your free auth token from https://dashboard.ngrok.com/get-started/your-authtoken

from getpass import getpass
from pyngrok import ngrok
import os

NGROK_AUTH_TOKEN = getpass("Enter your ngrok auth token: ").strip()
if not NGROK_AUTH_TOKEN:
    raise ValueError("An ngrok auth token is required. Get one from https://dashboard.ngrok.com/get-started/your-authtoken")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Close an older tunnel from a previous run, if present.
try:
    ngrok.kill()
except Exception:
    pass

tunnel = ngrok.connect(addr=8501, proto="http")
url = tunnel.public_url

print("🚀 Open your portfolio using this ngrok URL:")
print(url)
